# 01 - Data exploration and baseline

**Goal:** look at the dataset, tag Hinglish rows, and get a simple TF-IDF + Logistic Regression baseline.
The baseline score is the number the transformers have to beat.

Labels: `0` = negative, `1` = neutral, `2` = positive.

In [ ]:
%pip install -q ftfy tweet-preprocessor

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/23f2004336/hinglish-sentiment.git"

# Make `src/` importable: works locally (inside notebooks/ or the repo root) and on Kaggle.
if os.path.exists("../src"):
    sys.path.insert(0, "..")
    RESULTS_DIR = "../results"
elif os.path.exists("src"):
    sys.path.insert(0, ".")
    RESULTS_DIR = "results"
else:
    if not os.path.exists("hinglish-sentiment"):
        subprocess.run(["git", "clone", REPO_URL], check=True)   # needs Internet ON in Kaggle
    sys.path.insert(0, os.path.abspath("hinglish-sentiment"))
    RESULTS_DIR = "."

## Load, clean and tag the data

In [ ]:
import json
import pandas as pd
from src.data import load_data

dfs = load_data()
for split, df in dfs.items():
    print(f"{split:<11} {len(df):>6} rows | hinglish: {int(df.hinglish.sum())}")

train, val, test = dfs["train"], dfs["validation"], dfs["test"]

## Label mix per slice

English is roughly balanced, but Hinglish is mostly neutral. That is why we report **Macro F1** and not accuracy
(always predicting "neutral" on Hinglish would already give about 58% accuracy).

In [ ]:
print(train.groupby("hinglish").label.value_counts(normalize=True).round(2).unstack())

## Spot-check the Hinglish tag
Read a few rows from each side. A few misses are fine.

In [ ]:
print("TAGGED as Hinglish:")
for t in train[train.hinglish].text.sample(10, random_state=1): print("-", t[:120])
print("\nNOT tagged:")
for t in train[~train.hinglish].text.sample(10, random_state=1): print("-", t[:120])

## Baseline: TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(vec.fit_transform(train.text), train.label)

pred = clf.predict(vec.transform(test.text))
y, m = test.label.values, test.hinglish.values

res = {
    "model": "tfidf_logreg",
    "overall": f1_score(y, pred, average="macro"),
    "hinglish": f1_score(y[m], pred[m], average="macro"),
    "english": f1_score(y[~m], pred[~m], average="macro"),
}
print({k: (round(v, 3) if not isinstance(v, str) else v) for k, v in res.items()})
print("\nHinglish only:\n", classification_report(y[m], pred[m]))

with open(os.path.join(RESULTS_DIR, "baseline_results.json"), "w") as f:
    json.dump(res, f, indent=2)

## Notes
- The Hinglish slice is small (about 485 test rows), so scores on it are noisy.
- Hinglish rows are almost all tweets; English rows are mostly reviews. So "Hinglish" here also means "tweets".